In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
from pathlib import Path

import torch
import zarr
from zarr.codecs import BloscCodec
from PIL import Image
from tqdm.auto import tqdm
import pyvista as pv
import matplotlib
matplotlib.use("Agg") if os.environ.get("PYVISTA_OFF_SCREEN") else None
%matplotlib inline

pv.set_jupyter_backend("static" if os.environ.get("PYVISTA_OFF_SCREEN") else "trame")

from collab_splats.pointcloud.feedforward.base import FeedforwardResult
from collab_splats.semantics.features import MaskCLIPExtractor, DINOFeatureExtractor, Talk2DinoExtractor
from collab_splats.semantics.compression import FeatureAutoencoder
from collab_splats.pointcloud.utils import lift_features
from collab_splats.utils.visualization import pointcloud_to_polydata

## §0 — Configuration

In [ ]:
%run ../tutorial_config.py

# ── Configuration ─────────────────────────────────────────────────────────────
METHOD         = "vggtx"    # "vggtx" | "mapanything"
LATENT_DIM     = 13
QUERY_POSITIVE = ["tree"]
QUERY_NEGATIVE = ["ground"]
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

# Prefer BA reconstruction if available; fall back to raw feedforward result
_ba   = CACHE_DIR / METHOD / "ba" / "reconstruction.zarr"
_raw  = CACHE_DIR / METHOD / "reconstruction.zarr"
RECON = _ba if _ba.exists() else _raw
_base = RECON.parent  # directory that holds all lifted.zarr outputs for this variant

# Extractor-named cache paths — never collide when switching extractors or methods
LIFTED_MASKCLIP  = _base / "lifted_maskclip.zarr"
LIFTED_TALK2DINO = _base / "lifted_talk2dino.zarr"

assert RECON.exists(), (
    f"Reconstruction not found at {RECON}. "
    "Run 02_pointcloud/feedforward_methods first."
)
print(f"Device: {DEVICE}  |  RECON: {RECON}")
print(f"BA: {'yes' if _ba.exists() else 'no — using raw feedforward result'}")
print(f"MaskCLIP cache:  {LIFTED_MASKCLIP}")
print(f"Talk2DINO cache: {LIFTED_TALK2DINO}")

## §1 — Load Reconstruction

Loads the cached feedforward reconstruction from zarr. Produces a `FeedforwardResult` with
world-space 3D points, per-point colors, pixel source indices, and source image paths.
Run `02_pointcloud/feedforward_methods` first if the assertion above failed.

In [ ]:
# Load reconstruction from zarr cache
out = FeedforwardResult.load_zarr(RECON)
print(f"Loaded reconstruction: {out.pts3d.shape[0]:,} pts  |  {len(out.image_paths)} frames")

## §2 — Inspect Outputs

Confirms the three arrays needed for feature lifting: `pts3d` (3D positions), `pixel_indices`
(source frame/row/col per point), and `colors` (RGB). `pixel_indices` is the bridge between
the 2D feature maps and the 3D point cloud.

In [ ]:
pts3d         = out.pts3d          # (P, 3) float32
pixel_indices = out.pixel_indices  # (P, 3) int32  [frame_id, row, col]
colors        = out.colors         # (P, 3) uint8

print(f"pts3d:         {pts3d.shape}")
print(f"pixel_indices: {pixel_indices.shape}")
print(f"colors:        {colors.shape}")
print(f"image_paths:   {len(out.image_paths)} frames")

## §3 — MaskCLIP: Extract, Compress & Lift

Extracts MaskCLIP patch features from every source frame, then fits a `FeatureAutoencoder`
to compress them from full-dim to `LATENT_DIM` before lifting onto the point cloud.

MaskCLIP patch features are purely appearance-based — they lack structural grounding.
A DINOv2 regularization branch during AE training improves the latent space geometry
by pulling structurally similar patches together. Talk2DINO (§6) already carries CLIP
grounding, so no regularization is needed there.

**Cache hit:** `feat_mc` is loaded directly — extraction and compression are skipped.

In [ ]:
# Load all source frames once — used by both extractor paths
imgs       = [Image.open(p).convert("RGB") for p in tqdm(out.image_paths, desc="Loading frames")]
image_size = (imgs[0].height, imgs[0].width)

########################################################################

if LIFTED_MASKCLIP.exists():
    # Cache hit — load compressed-decoded features directly
    _s      = zarr.open(str(LIFTED_MASKCLIP), mode="r")
    feat_mc = torch.from_numpy(np.asarray(_s["features"][:]))
    print(f"Loaded from cache: {LIFTED_MASKCLIP}  shape={feat_mc.shape}")
else:
    # Init MaskCLIP (semantics) and DINOv2 (AE regularization only)
    maskclip = MaskCLIPExtractor(device=DEVICE)
    dinov2   = DINOFeatureExtractor(device=DEVICE)

    # Extract per-frame feature maps
    mc_maps = maskclip.forward(imgs)  # list of (D_mc, H_p, W_p)
    dv_maps = dinov2.forward(imgs)    # list of (384, H_p, W_p)

    # Stack all patch vectors for AE training
    D, H_p, W_p = mc_maps[0].shape
    mc_patches = torch.cat([fm.permute(1, 2, 0).reshape(-1, D)   for fm in mc_maps])
    dv_patches = torch.cat([fm.permute(1, 2, 0).reshape(-1, 384) for fm in dv_maps])

    # Fit AE: MaskCLIP reconstruction + DINOv2 regularization branch
    ae = FeatureAutoencoder(
        input_dim=D, latent_dim=LATENT_DIM,
        regularization_kwargs={"branches": {"dinov2": 384}, "weight": 0.1},
    )
    ae.fit(mc_patches.to(DEVICE), reg_targets={"dinov2": dv_patches.to(DEVICE)})

    # Compress each frame map to LATENT_DIM
    compressed_mc = [
        ae.per_point_encode(fm.permute(1, 2, 0).reshape(-1, D).to(DEVICE))
           .detach().cpu()
           .reshape(H_p, W_p, LATENT_DIM)
           .permute(2, 0, 1)
        for fm in mc_maps
    ]

    # Lift compressed codes to 3D points, then decode to full-dim
    codes_mc = lift_features(compressed_mc, out.pixel_indices, image_size=image_size)
    feat_mc  = ae.per_point_decode(codes_mc.to(DEVICE)).detach().cpu()  # (P, D)

    # Save to cache
    LIFTED_MASKCLIP.parent.mkdir(parents=True, exist_ok=True)
    _lz4 = BloscCodec(cname="lz4")
    _s   = zarr.open(str(LIFTED_MASKCLIP), mode="w")
    _s.create_array("features", data=feat_mc.numpy(), chunks=feat_mc.shape, compressors=_lz4)
    print(f"Lifted & saved: {feat_mc.shape}  →  {LIFTED_MASKCLIP}")

print(f"feat_mc: {feat_mc.shape}")

## §4 — MaskCLIP: Text Queries → Per-Point Scores

Scores each 3D point against the configured text queries. `score_queries` returns a
contrastive softmax score in [0, 1] — higher means the point matches the positive queries.
`compute_similarity` returns raw cosine similarities per query for multi-query inspection.

In [ ]:
# Init MaskCLIP extractor for text scoring (needed even on cache hit)
maskclip_scorer = MaskCLIPExtractor(device=DEVICE)

# Move features to device
feat_mc_dev = feat_mc.to(DEVICE)

# Contrastive score across all positive queries vs negative queries
scores_mc = maskclip_scorer.score_queries(
    feat_mc_dev, positive=QUERY_POSITIVE, negative=QUERY_NEGATIVE, temperature=0.05
).detach().cpu().numpy()

# Per-query cosine similarities (P, Q)
per_query_mc = maskclip_scorer.compute_similarity(feat_mc_dev, QUERY_POSITIVE).detach().T.cpu().numpy()

print(f"scores_mc:    {scores_mc.shape}  min={scores_mc.min():.3f}  max={scores_mc.max():.3f}")
print(f"per_query_mc: {per_query_mc.shape}")

## §5 — MaskCLIP: Interactive 3D Viewer

Attaches semantic scores and per-query arrays to the point cloud. Use the scalar selector
in the side panel to switch between the RGB view and each semantic query.

In [ ]:
cloud_mc = pointcloud_to_polydata(
    pts3d,
    RGB=colors,
    semantic=scores_mc,
    **{q.replace(" ", "_"): per_query_mc[:, i] for i, q in enumerate(QUERY_POSITIVE)},
)

pl = pv.Plotter(title="MaskCLIP Semantic Lifting")
pl.add_mesh(cloud_mc, scalars="semantic", cmap="plasma", point_size=2)
pl.add_scalar_bar("semantic score", fmt="%.2f")
pl.add_title("MaskCLIP + DINOv2 reg — switch scalars in side panel", font_size=9)
pl.show()

## §6 — Talk2DINO: Extract, Compress & Lift

Runs the same extract → compress → lift pipeline with Talk2DINO patch features.
Talk2DINO is CLIP-grounded — its patch features already carry structural and semantic
information from CLIP pre-training. No DINOv2 regularization branch is needed;
the AE trains on Talk2DINO patches directly.

**Cache hit:** `feat_t2d` is loaded directly — extraction and compression are skipped.

In [ ]:
if LIFTED_TALK2DINO.exists():
    # Cache hit — load compressed-decoded features directly
    _s       = zarr.open(str(LIFTED_TALK2DINO), mode="r")
    feat_t2d = torch.from_numpy(np.asarray(_s["features"][:]))
    print(f"Loaded from cache: {LIFTED_TALK2DINO}  shape={feat_t2d.shape}")
else:
    # Init Talk2DINO — no DINOv2 needed (already CLIP-grounded)
    talk2dino = Talk2DinoExtractor(device=DEVICE)

    # Extract per-frame feature maps
    t2d_maps = talk2dino.forward(imgs)  # list of (D_t2d, H_p, W_p)

    # Stack all patch vectors for AE training
    D_t, H_p, W_p = t2d_maps[0].shape
    t2d_patches = torch.cat([fm.permute(1, 2, 0).reshape(-1, D_t) for fm in t2d_maps])

    # Fit AE: plain reconstruction, no regularization branch
    ae_t2d = FeatureAutoencoder(input_dim=D_t, latent_dim=LATENT_DIM)
    ae_t2d.fit(t2d_patches.to(DEVICE))

    # Compress each frame map to LATENT_DIM
    compressed_t2d = [
        ae_t2d.per_point_encode(fm.permute(1, 2, 0).reshape(-1, D_t).to(DEVICE))
              .detach().cpu()
              .reshape(H_p, W_p, LATENT_DIM)
              .permute(2, 0, 1)
        for fm in t2d_maps
    ]

    # Lift compressed codes to 3D points, then decode to full-dim
    codes_t2d = lift_features(compressed_t2d, out.pixel_indices, image_size=image_size)
    feat_t2d  = ae_t2d.per_point_decode(codes_t2d.to(DEVICE)).detach().cpu()  # (P, D_t)

    # Save to cache
    LIFTED_TALK2DINO.parent.mkdir(parents=True, exist_ok=True)
    _lz4 = BloscCodec(cname="lz4")
    _s   = zarr.open(str(LIFTED_TALK2DINO), mode="w")
    _s.create_array("features", data=feat_t2d.numpy(), chunks=feat_t2d.shape, compressors=_lz4)
    print(f"Lifted & saved: {feat_t2d.shape}  →  {LIFTED_TALK2DINO}")

print(f"feat_t2d: {feat_t2d.shape}")

## §7 — Talk2DINO: Text Queries → Per-Point Scores

Same query scoring as §4 but using the Talk2DINO text encoder. Both extractors implement
`score_queries` and `compute_similarity` via `BaseQueryableExtractor`.

In [ ]:
# Init Talk2DINO scorer (needed even on cache hit)
talk2dino_scorer = Talk2DinoExtractor(device=DEVICE)

feat_t2d_dev = feat_t2d.to(DEVICE)

scores_t2d = talk2dino_scorer.score_queries(
    feat_t2d_dev, positive=QUERY_POSITIVE, negative=QUERY_NEGATIVE, temperature=0.05
).detach().cpu().numpy()

per_query_t2d = talk2dino_scorer.compute_similarity(feat_t2d_dev, QUERY_POSITIVE).detach().T.cpu().numpy()

print(f"scores_t2d:    {scores_t2d.shape}  min={scores_t2d.min():.3f}  max={scores_t2d.max():.3f}")
print(f"per_query_t2d: {per_query_t2d.shape}")

## §8 — Side-by-Side Comparison

Renders both semantic point clouds in linked panels. Orbiting one view orbits both.
The contrast between MaskCLIP (DINOv2-regularized) and Talk2DINO highlights the
effect of extractor choice on 3D semantic quality.

In [ ]:
cloud_t2d = pointcloud_to_polydata(
    pts3d,
    RGB=colors,
    semantic=scores_t2d,
    **{q.replace(" ", "_"): per_query_t2d[:, i] for i, q in enumerate(QUERY_POSITIVE)},
)

pl = pv.Plotter(shape=(1, 2), title="MaskCLIP vs Talk2DINO — Semantic Lifting")
pl.subplot(0, 0)
pl.add_mesh(cloud_mc.copy(), scalars="semantic", cmap="plasma", point_size=2)
pl.add_title("MaskCLIP + DINOv2 reg", font_size=10)
pl.subplot(0, 1)
pl.add_mesh(cloud_t2d.copy(), scalars="semantic", cmap="plasma", point_size=2)
pl.add_title("Talk2DINO", font_size=10)
pl.link_views()
pl.show()